In [1]:
import pandas as pd
import xarray as xr
import numpy as np

import geopandas as gpd
import rioxarray

In [2]:
def compute_state_precip_anoms(precip_anom_ds, states_gdf, time_dim='time', state_col='STATEFP'):
    """
    Compute average precip anomaly per state and time using rioxarray vectorized clipping.

    Parameters:
        precip_anom_ds: xarray DataArray or Dataset (with dims time, lat, lon) and CRS defined
        states_gdf: GeoDataFrame with state polygons and a column named state_col
        time_dim: name of the time dimension
        state_col: column in states_gdf identifying the states

    Returns:
        pd.DataFrame with columns: DATE, STATE, PRECIP_ANOM
    """

    # Ensure CRS on precip data
    if precip_anom_ds.rio.crs is None:
        precip_anom_ds = precip_anom_ds.rio.write_crs("EPSG:4326")
        
    precip_anom_ds = precip_anom_ds.rename({'lon': 'x', 'lat': 'y'})

    # Reproject states to precip CRS if needed
    if states_gdf.crs != precip_anom_ds.rio.crs:
        states_gdf = states_gdf.to_crs(precip_anom_ds.rio.crs)

    print(states_gdf)
    
    records = []

    for idx, state_row in states_gdf.iterrows():
        state_id = state_row[state_col]
        geom = [state_row.geometry]  # .rio.clip expects list of geometries

        # Clip raster to state polygon for all times
        clipped = precip_anom_ds.rio.clip(geom, precip_anom_ds.rio.crs, drop=False, invert=False)
        print(clipped)
        
        # Calculate spatial mean for each time ignoring NaNs (outside polygon)
        state_means = clipped.mean(dim=['y', 'x'], skipna=True).compute()
        print(state_means)
        
        # Build records for DataFrame
        for time_val, mean_val in zip(state_means[time_dim].values, state_means.values):
            records.append({'DATE': pd.to_datetime(str(time_val)), 'STATE': state_id, 'PRECIP_ANOM': mean_val})

    df_result = pd.DataFrame(records)
    return df_result

In [3]:
# Fatality Analysis Reporting System Filtered to include only weather related crashes
df_fars = pd.read_csv('../data/FARS/DATABASE3.csv')

# States shapefile
states_map = gpd.read_file('../data/state_shape_file/cb_2018_us_state_20m.shp')

In [4]:
#df_enso = pd.read_csv('../data/ENSO/ENSOMONTHLY.csv')
#df_swe = pd.read_csv('../data/swe/processed/processed_swe_data.csv')
#ds_comp = xr.open_dataset('../data/wxregimes/era5_cluster_comp_z_na_DJF1981-2019.nc')
#ds_pcomp = xr.open_dataset('../data/wxregimes/era5_cluster_comp_p_na_DJF1981-2019.nc')
#states = gpd.read_file('/data/esplab/aelyoussoufi/cb_2018_us_state_20m.shp')

# DAILY DATABASE

In [5]:
# Cluster definitions from Weather Regime analysis
ds_cd = xr.open_dataset('../data/wxregimes/kmeans_4cluster_DJF_1981-2019_NA.nc')

## 1. Read Weather-Related Crash Database

In [6]:
# Read Weather-Related Crash Database
df_fars = pd.read_csv('../data/FARS/DATABASE3.csv')

# Convert date
df_fars['DATE'] = pd.to_datetime(df_fars[['YEAR', 'MONTH', 'DAY']], errors='coerce')
df_fars = df_fars.dropna(subset=['DATE'])

# Filter to CONUS (exclude AK, HI, PR)
df_fars = df_fars[~df_fars['STATE'].isin(['02', '15', '72'])]

## 2. Get state precip anomalies

fname_precip='/data/esplab/shared/obs/gridded/atm/precip/daily/chirps-v2.0/p05/*'
ds_chirps=xr.open_mfdataset(fname_precip,combine='by_coords')
ds_chirps = ds_chirps.rename({'latitude':'lat','longitude':'lon'})
ds_chirps=ds_chirps.assign_coords(lon=((ds_chirps['lon'] + 360) % 360))
ds_chirps=ds_chirps.sortby(ds_chirps['lon'])

ds_chirps

states_map

df_precip_state=compute_state_precip_anoms(ds_chirps, states_map, time_dim='time')

## 3. Get cluster/regimes information

In [ ]:
# Subset DJF
ds_djf = ds_cd.sel(time=ds_cd['time.month'].isin([12, 1, 2]))

# Convert to DataFrame
df_regime = ds_djf[['cluster']].to_dataframe().reset_index()

# Rename and format
df_regime = df_regime.rename(columns={'time': 'DATE', 'cluster': 'REGIME'})
df_regime['DATE'] = pd.to_datetime(df_regime['DATE'])
df_regime['REGIME'] = df_regime['REGIME'].astype(int)

## 4. Create Daily Database

In [ ]:
# Ensure DATE is datetime and normalized
df_fars['DATE'] = pd.to_datetime(df_fars[['YEAR', 'MONTH', 'DAY']], errors='coerce')
df_fars['DATE'] = df_fars['DATE'].dt.normalize()

# Aggregate daily crash counts by state
daily_df = (
    df_fars.groupby(['DATE', 'STATE'])
    .size().reset_index(name='CRASH_COUNT')
)
daily_df['DOY'] = daily_df['DATE'].dt.dayofyear

# Daily climatology (mean and std for each state/day-of-year)
climatology = (
    daily_df.groupby(['STATE', 'DOY'])['CRASH_COUNT']
    .agg(CLIM_MEAN='mean', CLIM_STD='std')
    .reset_index()
)

# Merge climatology
daily_df = daily_df.merge(climatology, on=['STATE', 'DOY'], how='left')

# Recalculate anomalies with std-normalization
daily_df['CRASH_ANOMALY'] = daily_df['CRASH_COUNT'] - daily_df['CLIM_MEAN']
daily_df['CRASH_ANOM_NORM'] = daily_df['CRASH_ANOMALY'] / daily_df['CLIM_STD']

# Normalize and filter daily_df dates
daily_df['DATE'] = pd.to_datetime(daily_df['DATE']).dt.floor('D')
daily_df = daily_df[(daily_df['DATE'].dt.year >= 1981) & (daily_df['DATE'].dt.year <= 2019)]

# Normalize and filter df_regime dates
df_regime['DATE'] = pd.to_datetime(df_regime['DATE']).dt.floor('D')
df_regime = df_regime[(df_regime['DATE'].dt.year >= 1981) & (df_regime['DATE'].dt.year <= 2019)]

# Merge regime info
daily_df = daily_df.merge(df_regime, on='DATE', how='left')

# Now filter daily_df to DJF months only (if desired)
daily_df = daily_df[daily_df['DATE'].dt.month.isin([12, 1, 2])]

# Normalize df_regime
df_regime['DATE'] = pd.to_datetime(df_regime['DATE']).dt.normalize()

# Merge REGIME info first
if 'REGIME' in daily_df.columns:
    daily_df = daily_df.drop(columns=['REGIME'])
daily_df = daily_df.merge(df_regime, on='DATE', how='left')

# Now filter to DJF
daily_df = daily_df[daily_df['DATE'].dt.month.isin([12, 1, 2])]

# Add state abbreviation
daily_df['STATE'] = daily_df['STATE'].astype(str).str.zfill(2)
states_map['STATEFP'] = states_map['STATEFP'].astype(str).str.zfill(2)
state_abbr = states_map[['STATEFP', 'STUSPS']].drop_duplicates()
daily_df = daily_df.merge(state_abbr, left_on='STATE', right_on='STATEFP', how='left')

# Merge on DATE and STATE
daily_df = daily_df.merge(df_precip_state, on=['DATE', 'STATE'], how='left')

# Clean up
daily_df = daily_df.drop(columns=['STATE', 'STATEFP'])
daily_df = daily_df.rename(columns={'STUSPS': 'STATE'})

In [ ]:
daily_df

In [ ]:
print(df_regime.head())
print(df_regime.dtypes)